In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/Nvu/knowledge_base')

In [3]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Load KB
kb_df = pd.read_parquet('knowledge_base.parquet')

with open('knowledge_base.json', 'r', encoding='utf-8') as f:
    kb_json = json.load(f)

print(f"KB Statistics:")
print(f"   • Total records (flat): {len(kb_df)}")
print(f"   • Unique QA pairs: {kb_df['qa_id'].nunique()}")
print(f"   • Unique contexts: {kb_df['context_text'].nunique()}")
print(f"   • Avg contexts per QA: {len(kb_df) / kb_df['qa_id'].nunique():.1f}")
print(f"   • Avg BM25 score: {kb_df['bm25_score'].mean():.2f}")

KB Statistics:
   • Total records (flat): 30045
   • Unique QA pairs: 10015
   • Unique contexts: 14201
   • Avg contexts per QA: 3.0
   • Avg BM25 score: 55.97


In [5]:
def compute_kb_metrics(kb_df):
    """
    Tính các metrics tự động
    """
    metrics = {}

    # 1. Context diversity (số contexts unique / total)
    metrics['context_diversity'] = kb_df['context_text'].nunique() / len(kb_df)

    # 2. Score distribution
    metrics['avg_bm25_score'] = kb_df['bm25_score'].mean()
    metrics['min_bm25_score'] = kb_df['bm25_score'].min()
    metrics['max_bm25_score'] = kb_df['bm25_score'].max()

    # 3. Context length stats
    kb_df['context_length'] = kb_df['context_text'].str.len()
    metrics['avg_context_length'] = kb_df['context_length'].mean()

    # 4. Coverage (passages có score > threshold)
    threshold = kb_df['bm25_score'].quantile(0.5)
    metrics['high_quality_ratio'] = (kb_df['bm25_score'] > threshold).mean()

    # 5. Multi-hop potential (contexts từ nhiều sources)
    sources_per_qa = kb_df.groupby('qa_id')['context_source_url'].nunique()
    metrics['avg_sources_per_qa'] = sources_per_qa.mean()
    metrics['multi_source_rate'] = (sources_per_qa > 1).mean()

    return metrics

metrics = compute_kb_metrics(kb_df)

print("\n📈 KNOWLEDGE BASE QUALITY METRICS:\n")
for key, value in metrics.items():
    print(f"   • {key}: {value:.3f}")


📈 KNOWLEDGE BASE QUALITY METRICS:

   • context_diversity: 0.473
   • avg_bm25_score: 55.967
   • min_bm25_score: 10.678
   • max_bm25_score: 501.552
   • avg_context_length: 1742.771
   • high_quality_ratio: 0.500
   • avg_sources_per_qa: 2.777
   • multi_source_rate: 0.979


- Vì dữ liệu hiện tại chưa được gán nhãn để tự huấn luyện mô hình NER nên t sử dụng model NER trên dữ liệu y tế tiếng Việt trên huggingface (transfer learning)
- Link: https://huggingface.co/leduckhai/VietMed-NER (paper publish ở NAACL rank A)
- Bài báo này nói về dataset và dùng các model để evaluate, t thấy XLM-Roberta-large có result tốt nhất nên dùng model này để NER trên dữ liệu của mình thử
- Có thể mình cần phải re-check lại kết quả trên 1 subset

In [6]:
# Cài đặt transformers và dependencies
# !pip install transformers torch sentencepiece

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline
)
import pandas as pd
import numpy as np
from pprint import pprint

In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "leduckhai/VietMed-NER"
subfolder = "xlm-roberta-large-VietMed-NER"

print(f"🔄 Đang tải model từ: {model_name}/{subfolder}...")

# Load tokenizer từ subfolder
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    subfolder=subfolder,
    trust_remote_code=True  # Nếu cần custom code
)

# Load model từ subfolder
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    subfolder=subfolder,
    trust_remote_code=True
)

# Chuyển sang eval mode
model.eval()

print("✅ Model loaded successfully!")
print(f"\n📊 Model Config:")
print(f"   • Number of labels: {model.config.num_labels}")
print(f"   • Model type: {model.config.model_type}")
print(f"   • Hidden size: {model.config.hidden_size}")

# Xem entity labels
if hasattr(model.config, 'id2label'):
    print(f"\n🏷️ Entity Types:")
    for id, label in sorted(model.config.id2label.items()):
        print(f"   {id:2d}: {label}")
else:
    print("\n⚠️ Warning: id2label not found in config")

🔄 Đang tải model từ: leduckhai/VietMed-NER/xlm-roberta-large-VietMed-NER...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

xlm-roberta-large-VietMed-NER/sentencepi(…):   0%|          | 0.00/5.07M [00:00<?, ?B/s]

xlm-roberta-large-VietMed-NER/tokenizer.(…):   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

xlm-roberta-large-VietMed-NER/pytorch_mo(…):   0%|          | 0.00/2.24G [00:00<?, ?B/s]

✅ Model loaded successfully!

📊 Model Config:
   • Number of labels: 37
   • Model type: xlm-roberta
   • Hidden size: 1024

🏷️ Entity Types:
    0: I-ORGAN
    1: B-PREVENTIVEMED
    2: B-DISEASESYMTOM
    3: B-FOODDRINK
    4: B-ORGANIZATION
    5: B-OCCUPATION
    6: B-DRUGCHEMICAL
    7: I-FOODDRINK
    8: I-DISEASESYMTOM
    9: I-UNITCALIBRATOR
   10: I-DATETIME
   11: I-DIAGNOSTICS
   12: B-TRANSPORTATION
   13: B-GENDER
   14: B-AGE
   15: B-DATETIME
   16: B-LOCATION
   17: B-TREATMENT
   18: I-DRUGCHEMICAL
   19: I-PREVENTIVEMED
   20: I-TREATMENT
   21: I-ORGANIZATION
   22: 0
   23: B-UNITCALIBRATOR
   24: B-MEDDEVICETECHNIQUE
   25: I-OCCUPATION
   26: B-PERSONALCARE
   27: I-PERSONALCARE
   28: I-GENDER
   29: I-SURGERY
   30: I-AGE
   31: I-LOCATION
   32: B-DIAGNOSTICS
   33: I-MEDDEVICETECHNIQUE
   34: B-ORGAN
   35: B-SURGERY
   36: I-TRANSPORTATION


In [40]:
import pandas as pd
import numpy as np
import json
import re
from tqdm import tqdm
from collections import Counter, defaultdict
from datetime import datetime

# Config
MEDICAL_ENTITY_TYPES = {
    'DISEASESYMTOM', 'DRUGCHEMICAL', 'TREATMENT', 'SURGERY',
    'ORGAN', 'DIAGNOSTICS', 'MEDDEVICETECHNIQUE', 'PREVENTIVEMED'
}

CONFIDENCE_THRESHOLDS = {
    'question': 0.7,
    'answer': 0.8,
    'context': 0.9
}

MIN_LENGTH = 3
BATCH_SIZE = 16
MAX_TEXT_LENGTH = 2000

In [34]:
def extract_medical_entities_from_text(text, ner_pipeline, min_confidence=0.9):
    """
    Extract medical entities từ 1 text

    Args:
        text: Input text
        ner_pipeline: HF NER pipeline
        min_confidence: Minimum confidence threshold

    Returns:
        List of dict với keys: word, type, score
    """
    # Truncate
    text = text[:MAX_TEXT_LENGTH]

    # Predict
    try:
        raw = ner_pipeline(text)
    except Exception as e:
        print(f"⚠️ Error during NER: {e}")
        return []

    # Filter & clean
    entities = []

    for e in raw:
        # Remove B-/I- prefix
        etype = e['entity_group'].replace('B-', '').replace('I-', '')

        # Filter medical types only
        if etype not in MEDICAL_ENTITY_TYPES:
            continue

        # Filter confidence
        if e['score'] < min_confidence:
            continue

        # Extract word
        if e.get('start') is not None and e.get('end') is not None:
            word = text[e['start']:e['end']]
        else:
            word = e['word']

        # Clean
        word = word.replace('▁', ' ').strip()
        word = re.sub(r'^[^\w\s\u00C0-\u1EF9]+|[^\w\s\u00C0-\u1EF9]+$', '', word)
        word = re.sub(r'\s+', ' ', word).strip()

        # Validate
        if len(word) < MIN_LENGTH:
            continue

        # Skip pattern errors
        if re.match(r'^[a-zA-Z]\s', word):
            continue

        entities.append({
            'word': word,
            'type': etype,
            'score': float(e['score'])
        })

    # Deduplicate (keep highest score)
    seen = {}
    for entity in entities:
        key = entity['word'].lower()
        if key not in seen or entity['score'] > seen[key]['score']:
            seen[key] = entity

    return list(seen.values())

In [35]:
def extract_entities_from_qa_record(row, ner_pipeline):
    """
    Extract entities từ 1 QA record (question + answer + context)
    """
    result = {
        'qa_id': row['qa_id'],
        'question': row['question'],
        'answer': row['answer'],
        'context_text': row['context_text'],
        'context_title': row['context_title'],
        'context_url': row.get('context_source_url', ''),
        'entities': {
            'question': [],
            'answer': [],
            'context': []
        }
    }

    # Extract với confidence khác nhau cho từng source
    if row['question']:
        result['entities']['question'] = extract_medical_entities_from_text(
            row['question'],
            ner_pipeline,
            min_confidence=CONFIDENCE_THRESHOLDS['question']
        )

    if row['context_text']:
        result['entities']['context'] = extract_medical_entities_from_text(
            row['context_text'],
            ner_pipeline,
            min_confidence=CONFIDENCE_THRESHOLDS['context']
        )

    if row['answer']:
        result['entities']['answer'] = extract_medical_entities_from_text(
            row['answer'],
            ner_pipeline,
            min_confidence=CONFIDENCE_THRESHOLDS['answer']
        )

    return result


def batch_extract_from_qa_dataset(kb_df, ner_pipeline, batch_size=16):
    """
    Extract entities từ toàn bộ dataset
    """
    all_results = []
    errors = []

    print(f"🔄 Processing {len(kb_df)} QA records...")
    print(f"📊 Batch size: {batch_size}\n")

    for i in tqdm(range(0, len(kb_df), batch_size), desc="Extracting"):
        batch = kb_df.iloc[i:i+batch_size]

        for idx, row in batch.iterrows():
            try:
                result = extract_entities_from_qa_record(row, ner_pipeline)
                all_results.append(result)
            except Exception as e:
                errors.append({
                    'row_index': idx,
                    'qa_id': row.get('qa_id', 'unknown'),
                    'error': str(e)
                })

    if errors:
        print(f"\n⚠️ {len(errors)} errors occurred during processing")
        print(f"   Check errors in returned dict")

    return {
        'results': all_results,
        'errors': errors
    }

In [36]:
def compute_statistics(results, entity_dict):
    """
    Compute statistics từ extraction results
    """
    stats = {
        'timestamp': datetime.now().isoformat(),
        'total_qa_records': len(results),
        'total_unique_entities': len(entity_dict),
    }

    # Count mentions per source
    mentions_by_source = {'question': 0, 'answer': 0, 'context': 0}
    for r in results:
        for src in ['question', 'answer', 'context']:
            mentions_by_source[src] += len(r['entities'][src])

    stats['mentions_by_source'] = mentions_by_source
    stats['total_mentions'] = sum(mentions_by_source.values())

    # Entity type distribution
    type_dist = Counter(ent['entity_type'] for ent in entity_dict.values())
    stats['entity_type_distribution'] = dict(type_dist)

    # Confidence stats
    all_scores = [ent['avg_confidence'] for ent in entity_dict.values()]
    stats['confidence'] = {
        'mean': float(np.mean(all_scores)),
        'median': float(np.median(all_scores)),
        'min': float(np.min(all_scores)),
        'max': float(np.max(all_scores))
    }

    # Top entities
    top_entities = sorted(
        entity_dict.items(),
        key=lambda x: x[1]['frequency'],
        reverse=True
    )[:20]

    stats['top_20_entities'] = [
        {
            'id': eid,
            'canonical': ent['canonical_form'],
            'type': ent['entity_type'],
            'frequency': ent['frequency']
        }
        for eid, ent in top_entities
    ]

    return stats

def print_statistics(stats):
    """
    Print statistics dễ đọc
    """
    print("\n" + "="*80)
    print("📊 NER EXTRACTION STATISTICS")
    print("="*80)

    print(f"\n📝 Dataset:")
    print(f"   • Total QA records: {stats['total_qa_records']}")
    print(f"   • Total entity mentions: {stats['total_mentions']}")
    print(f"   • Unique entities: {stats['total_unique_entities']}")

    print(f"\n📍 Mentions by source:")
    for src, count in stats['mentions_by_source'].items():
        pct = count / stats['total_mentions'] * 100 if stats['total_mentions'] > 0 else 0
        print(f"   • {src:10s}: {count:6d} ({pct:5.1f}%)")

    print(f"\n🏷️ Entity type distribution:")
    for etype, count in sorted(
        stats['entity_type_distribution'].items(),
        key=lambda x: x[1],
        reverse=True
    ):
        pct = count / stats['total_unique_entities'] * 100
        print(f"   • {etype:25s}: {count:5d} ({pct:5.1f}%)")

    print(f"\n🎯 Confidence scores:")
    print(f"   • Mean: {stats['confidence']['mean']:.3f}")
    print(f"   • Median: {stats['confidence']['median']:.3f}")
    print(f"   • Range: [{stats['confidence']['min']:.3f}, {stats['confidence']['max']:.3f}]")

    print(f"\n🔝 Top 10 most frequent entities:")
    for i, item in enumerate(stats['top_20_entities'][:10], 1):
        print(f"   {i:2d}. {item['canonical']:30s} ({item['type']:15s}) - {item['frequency']} times")
    print("="*80)

In [37]:
def build_entity_dictionary(results):
    """
    Build entity dictionary từ all mentions
    """
    entity_mentions = defaultdict(lambda: {
        'texts': [],
        'type': '',
        'scores': [],
        'sources': []
    })

    for record in results:
        for source_type in ['question', 'answer', 'context']:
            for entity in record['entities'][source_type]:
                key = (entity['word'].lower(), entity['type'])

                entity_mentions[key]['texts'].append(entity['word'])
                entity_mentions[key]['type'] = entity['type']
                entity_mentions[key]['scores'].append(entity['score'])
                entity_mentions[key]['sources'].append({
                    'qa_id': record['qa_id'],
                    'source_type': source_type,
                    'score': entity['score']
                })

    # Build entity dict
    entity_dict = {}
    entity_id = 1

    for (text_lower, etype), data in entity_mentions.items():
        canonical = Counter(data['texts']).most_common(1)[0][0]
        aliases = list(set(data['texts']) - {canonical})

        entity_dict[f"ent_{entity_id:05d}"] = {
            'canonical_form': canonical,
            'entity_type': etype,
            'aliases': aliases,
            'frequency': len(data['sources']),
            'avg_confidence': float(np.mean(data['scores'])),
            'sources': data['sources'][:10]
        }

        entity_id += 1

    return entity_dict

def save_entity_centric_format(results, output_prefix='entities'):
    """
    Save theo format entity-centric
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("\n💾 Saving results...")

    # Build entity dictionary
    print("🔨 Building entity dictionary...")
    entity_dict = build_entity_dictionary(results)

    # Save entity dict
    dict_file = f"{output_prefix}_dictionary_{timestamp}.json"
    with open(dict_file, 'w', encoding='utf-8') as f:
        json.dump(entity_dict, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: {dict_file}")

    # Save full results
    full_file = f"{output_prefix}_full_{timestamp}.json"
    with open(full_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: {full_file}")

    return {
        'dict_file': dict_file,
        'full_file': full_file,
        'entity_dict': entity_dict
    }

In [38]:
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

Device set to use cuda:0


In [42]:
# ============================================
# MAIN EXECUTION
# ============================================

print("="*80)
print("🏥 MEDICAL NER EXTRACTION PIPELINE")
print("="*80)

# Step 1: Load dataset
print("\n[Step 1/4] Loading dataset...")
kb_df_orig = pd.read_parquet('knowledge_base.parquet')
kb_df_rank1 = kb_df_orig[kb_df_orig['rank'] == 1].copy()
# Step 2: Extract entities
print("\n[Step 2/4] Extracting entities from dataset...")
print(f"   Config:")
print(f"   • Question confidence: {CONFIDENCE_THRESHOLDS['question']}")
print(f"   • Answer confidence: {CONFIDENCE_THRESHOLDS['answer']}")
print(f"   • Context confidence: {CONFIDENCE_THRESHOLDS['context']}")
print(f"   • Batch size: {BATCH_SIZE}")
print()

extraction_output = batch_extract_from_qa_dataset(
    kb_df=kb_df_rank1,
    ner_pipeline=ner_pipeline,
    batch_size=BATCH_SIZE
)

results = extraction_output['results']
errors = extraction_output['errors']

print(f"\nExtraction complete!")
print(f"   • Processed: {len(results)} records")
if errors:
    print(f"   • Errors: {len(errors)} records")

# Step 3: Save results
print("\n[Step 3/4] Saving results...")
saved_files = save_entity_centric_format(
    results=results,
    output_prefix='medical_entities'
)

# Step 4: Summary
print("\n[Step 4/4] Summary")
print("="*80)
print("PIPELINE COMPLETE!")
print("\nOutput files:")
print(f"   1. {saved_files['dict_file']}")
print(f"      → Entity dictionary (for KB/KG)")
print(f"   2. {saved_files['full_file']}")
print(f"      → Full extraction results (backup)")

🏥 MEDICAL NER EXTRACTION PIPELINE

[Step 1/4] Loading dataset...

[Step 2/4] Extracting entities from dataset...
   Config:
   • Question confidence: 0.7
   • Answer confidence: 0.8
   • Context confidence: 0.9
   • Batch size: 16

🔄 Processing 10015 QA records...
📊 Batch size: 16



Extracting: 100%|██████████| 626/626 [25:23<00:00,  2.43s/it]



Extraction complete!
   • Processed: 10015 records

[Step 3/4] Saving results...

💾 Saving results...
🔨 Building entity dictionary...
✅ Saved: medical_entities_dictionary_20251108_065852.json
✅ Saved: medical_entities_full_20251108_065852.json

[Step 4/4] Summary
PIPELINE COMPLETE!

Output files:
   1. medical_entities_dictionary_20251108_065852.json
      → Entity dictionary (for KB/KG)
   2. medical_entities_full_20251108_065852.json
      → Full extraction results (backup)


- Dict: entity-centric - mỗi entity là một entry (dùng để build KG, relation extraction)
- Full: document-centric - mỗi qa là một entry (để backup)

In [44]:
import numpy as np
# Compute statistics
print("Computing statistics...")
entity_dict = saved_files['entity_dict']
stats = compute_statistics(results, entity_dict)
print_statistics(stats)

Computing statistics...

📊 NER EXTRACTION STATISTICS

📝 Dataset:
   • Total QA records: 10015
   • Total entity mentions: 329095
   • Unique entities: 20481

📍 Mentions by source:
   • question  :  34847 ( 10.6%)
   • answer    : 100156 ( 30.4%)
   • context   : 194092 ( 59.0%)

🏷️ Entity type distribution:
   • DISEASESYMTOM            :  9578 ( 46.8%)
   • DRUGCHEMICAL             :  5061 ( 24.7%)
   • ORGAN                    :  2130 ( 10.4%)
   • MEDDEVICETECHNIQUE       :  1272 (  6.2%)
   • SURGERY                  :   990 (  4.8%)
   • DIAGNOSTICS              :   738 (  3.6%)
   • TREATMENT                :   436 (  2.1%)
   • PREVENTIVEMED            :   276 (  1.3%)

🎯 Confidence scores:
   • Mean: 0.966
   • Median: 0.985
   • Range: [0.701, 1.000]

🔝 Top 10 most frequent entities:
    1. điều trị                       (TREATMENT      ) - 10661 times
    2. khám                           (DIAGNOSTICS    ) - 6398 times
    3. tiêm                           (MEDDEVICETECHNIQUE